In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/train_final.csv")
test = pd.read_csv("../data/processed/test_final.csv")

X_train = train.drop("habitable", axis=1)
y_train = train["habitable"]

X_test = test.drop("habitable", axis=1)
y_test = test["habitable"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1486, 6)
Test shape: (372, 6)


In [2]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

class_weight_dict

{np.int64(0): np.float64(0.5027063599458728), np.int64(1): np.float64(92.875)}

In [3]:
X_train.isna().sum()

log_pl_orbsmax      0
log_st_lum          0
log_pl_rade         0
log_teq             0
log_st_teff         0
log_stellar_flux    0
dtype: int64

In [4]:
# Find rows with NaN
nan_rows = X_train[X_train.isna().any(axis=1)]

nan_rows

,log_pl_orbsmax,log_st_lum,log_pl_rade,log_teq,log_st_teff,log_stellar_flux


In [5]:
# Drop NaN rows
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

print("New train shape:", X_train.shape)

New train shape: (1486, 6)


In [6]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    class_weight=class_weight_dict,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully.")



Model trained successfully.


In [7]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

cm

array([[355,  15],
       [  0,   2]])

In [8]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98       370
           1       0.12      1.00      0.21         2

    accuracy                           0.96       372
   macro avg       0.56      0.98      0.59       372
weighted avg       1.00      0.96      0.98       372



In [9]:
y_probs = model.predict_proba(X_test)[:, 1]

y_probs[:10]

array([2.15873502e-02, 1.61611069e-02, 2.53612161e-01, 5.23735579e-12,
       2.22251695e-04, 1.66678727e-07, 2.56827703e-05, 8.01560215e-12,
       2.60314479e-09, 3.15098696e-01])

In [10]:
import pandas as pd

results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "probability": y_probs
})

results[results["predicted"] == 1]

,actual,predicted,probability
11,0,1,0.537964
55,0,1,0.974420
70,0,1,0.993761
79,0,1,0.580329
98,0,1,0.806448
119,0,1,0.893478
172,0,1,0.538759
191,1,1,0.829047
246,0,1,0.904518
275,0,1,0.678649


In [11]:
results["probability"].describe()

count    3.720000e+02
mean     4.793420e-02
std      1.736634e-01
min      6.505160e-20
25%      1.463141e-09
50%      1.676169e-06
75%      1.377640e-04
max      9.937610e-01
Name: probability, dtype: float64

In [12]:
class_weight="balanced"

In [13]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model_balanced.fit(X_train, y_train)

y_pred_bal = model_balanced.predict(X_test)

from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_bal)

array([[355,  15],
       [  0,   2]])

In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np

# Define model
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Stratified 5-fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use recall for class 1 as scoring metric
recall_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring="recall"
)

print("Recall scores per fold:", recall_scores)
print("Mean recall:", np.mean(recall_scores))

Recall scores per fold: [1. 1. 1. 1. 1.]
Mean recall: 1.0


In [16]:
precision_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring="precision"
)

print("Precision per fold:", precision_scores)
print("Mean precision:", np.mean(precision_scores))

Precision per fold: [0.33333333 0.11111111 0.2        0.25       0.33333333]
Mean precision: 0.24555555555555553


In [17]:
from sklearn.model_selection import cross_val_predict

# Get cross-validated predicted probabilities
y_probs_cv = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=skf,
    method="predict_proba"
)[:, 1]

y_probs_cv[:10]

array([1.39807377e-07, 3.75420222e-07, 1.74858374e-04, 5.79216792e-12,
       2.89592917e-05, 4.94750944e-04, 1.15495631e-06, 9.74544132e-06,
       9.64486937e-04, 2.71336157e-04])

In [18]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for t in thresholds:
    y_pred_t = (y_probs_cv >= t).astype(int)
    
    precision = precision_score(y_train, y_pred_t)
    recall = recall_score(y_train, y_pred_t)
    
    print(f"Threshold: {t}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print("-" * 30)

Threshold: 0.5
Precision: 0.235
Recall: 1.000
------------------------------
Threshold: 0.4
Precision: 0.211
Recall: 1.000
------------------------------
Threshold: 0.3
Precision: 0.174
Recall: 1.000
------------------------------
Threshold: 0.2
Precision: 0.154
Recall: 1.000
------------------------------
Threshold: 0.1
Precision: 0.110
Recall: 1.000
------------------------------
